In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [2]:
def readMariaDB(table_name, database):
    dfMaria = spark.read \
            .format("jdbc") \
            .option("driver", "org.mariadb.jdbc.Driver") \
            .option("url", f"jdbc:mysql://localhost:3306/{database}?permitMysqlScheme") \
            .option("dbtable", f"{table_name}") \
            .option("user", "root") \
            .option("password", "Thanhdan@123") \
            .load()

    return dfMaria

In [3]:
df = readMariaDB("order_items", "retails")

In [4]:
df.printSchema()

root
 |-- order_item_id: integer (nullable = true)
 |-- order_item_order_id: integer (nullable = true)
 |-- order_item_product_id: integer (nullable = true)
 |-- order_item_quantity: byte (nullable = true)
 |-- order_item_subtotal: float (nullable = true)
 |-- order_item_product_price: float (nullable = true)



In [5]:
df.createOrReplaceTempView("order_items")

# Exercise 7
Let us validate if we have an invalid order_item_subtotal as part of the order_items table.
* The order_items table has 6 fields.
* order_item_id
* order_item_order_id
* order_item_product_id
* order_item_quantity
* order_item_subtotal
* order_item_product_price
* order_item_subtotal is nothing but the product of order_item_quantity and order_item_product_price. It means order_item_subtotal is computed by multiplying order_item_quantity and order_item_product_price for each item.
* You need to get the count of order_items where order_item_subtotal is not equal to the
product of order_item_quantity and order_item_product_price.
* There can be issues related to rounding off. Make sure it is taken care of using the
appropriate function.

The output should be 0 as there are no such records.

In [6]:
spark.sql("""
    select count(*) as invalid_count
    from order_items
    where round(order_item_subtotal, 2) != round(cast(order_item_quantity as float) * order_item_product_price, 2)
""").show()

+-------------+
|invalid_count|
+-------------+
|            0|
+-------------+



# Exercise 8
Get the number of orders placed on weekdays and weekends in the month of January 2014.
* Orders have 4 fields: order_id, order_date, order_customer_id, order_status
* Use the order date to determine the day on which orders are placed.
* Output should contain 2 columns - day_type and order_count.
* day_type should have 2 values: Week days and Weekend days.

Here is the desired output.

In [7]:
df1 = readMariaDB("orders", "retails")
df1.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- order_customer_id: integer (nullable = true)
 |-- order_status: string (nullable = true)



In [8]:
df1.createOrReplaceTempView("orders")

In [9]:
spark.sql("""
    select
        case
            when weekday(order_date) >= 5 then 'Weekend days'
            else 'Week days' end as day_type,
        count(*) as order_count
    from orders
    group by day_type
""").show()

+------------+-----------+
|    day_type|order_count|
+------------+-----------+
|Weekend days|      19719|
|   Week days|      49164|
+------------+-----------+

